# Week 5 · Extra project

## Blackjack, object-oriented
Start from a function-based Blackjack (`deal_card()`, `calculate_score(cards)`, `compare(user_score, computer_score)`), then refactor: a `Hand` class with `self.cards`, `add_card(self)` and `score(self)` (ace 11 or 1, two-card 21 = blackjack), test `Hand` alone, use `user_hand` and `computer_hand`, rewrite the game loop, keep `compare` as a function.

### Version 1: functions (the starting point)

In [ ]:
import random

CARDS = [11, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10]   # 11 = ace, 10s = 10/J/Q/K


def deal_card():
    """Return a random card value."""
    return random.choice(CARDS)


def calculate_score(cards):
    """Return the score of a list of cards (0 means blackjack)."""
    if sum(cards) == 21 and len(cards) == 2:
        return 0
    total = sum(cards)
    aces = cards.count(11)
    while total > 21 and aces > 0:
        total -= 10          # count one ace as 1 instead of 11
        aces -= 1
    return total


def compare(user_score, computer_score):
    """Return the result message."""
    if user_score == computer_score:
        return "Draw"
    elif computer_score == 0:
        return "Lose, opponent has Blackjack"
    elif user_score == 0:
        return "Win with a Blackjack"
    elif user_score > 21:
        return "You went over. You lose"
    elif computer_score > 21:
        return "Opponent went over. You win"
    elif user_score > computer_score:
        return "You win"
    else:
        return "You lose"


user_cards = [deal_card(), deal_card()]
computer_cards = [deal_card(), deal_card()]

game_over = False
while not game_over:
    user_score = calculate_score(user_cards)
    print(f"Your cards: {user_cards}, current score: {user_score}")
    print(f"Computer's first card: {computer_cards[0]}")
    if user_score == 0 or user_score > 21 or calculate_score(computer_cards) == 0:
        game_over = True
    else:
        if input("Type 'y' to get another card, 'n' to pass: ") == "y":
            user_cards.append(deal_card())
        else:
            game_over = True

computer_score = calculate_score(computer_cards)
while computer_score != 0 and computer_score < 17:
    computer_cards.append(deal_card())
    computer_score = calculate_score(computer_cards)

print(f"Your final hand: {user_cards}, final score: {calculate_score(user_cards)}")
print(f"Computer's final hand: {computer_cards}, final score: {computer_score}")
print(compare(calculate_score(user_cards), computer_score))

> 💡 **Explanation**
>
> **Key points of version 1**
> - **The cards:** a list of numbers. The ace is 11, and J/Q/K are all 10. We put in four 10s so the chances are realistic.
> - **`calculate_score`:**
>   - First it checks for Blackjack (2 cards adding up to 21) and returns `0` for it. This is an agreed signal so that `compare` understands.
>   - Then the ace rule: while the total is over 21 and we still have an ace counted as 11, we count one of them as 1 instead (`-10`).
>   - `cards.count(11)` counts the aces.
> - **`compare`:** the order of the `elif`s matters (like D7 in Week 2): first the special cases (draw, blackjack, bust), then the normal comparison.
> - **The player's loop:** keeps taking cards until `game_over`. **The dealer's loop:** draws cards while under 17.
>
> **The problem with this version:** the list of cards (`user_cards`) and the functions that work on it are separate from each other. Every time, you have to send the list into the function: `calculate_score(user_cards)`. Version 2 fixes exactly this.

### Version 2: the `Hand` class (steps 2–5: build it and test it alone)

In [ ]:
import random

CARDS = [11, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10]


class Hand:
    """The cards held by one player, and their score."""

    def __init__(self):
        """Start with no cards."""
        self.cards = []

    def add_card(self):
        """Deal one random card into this hand."""
        self.cards.append(random.choice(CARDS))

    def score(self):
        """Return this hand's score (0 means blackjack)."""
        if sum(self.cards) == 21 and len(self.cards) == 2:
            return 0
        total = sum(self.cards)
        aces = self.cards.count(11)
        while total > 21 and aces > 0:
            total -= 10
            aces -= 1
        return total


# Step 5: test Hand on its own before using it in the game
test = Hand()
test.add_card()
test.add_card()
print(test.cards, test.score())

test.cards = [11, 11, 9]       # two aces: 11+11+9 = 31 -> 21
print(test.cards, test.score())

> 💡 **Explanation**
>
> **What moved?**
> - The list of cards is now an **attribute** of the object: `self.cards`. Each `Hand` has its own list (D2, Week 5).
> - `deal_card()` + `append` became the method `add_card(self)`: it adds a card to **its own** hand.
> - `calculate_score(cards)` became `score(self)`. It no longer takes a `cards` parameter; it reads `self.cards` itself. The logic is exactly the same.
>
> **Step 5, testing on its own:** before we touch the game, we test the class by itself. The second test has two aces on purpose: 11 + 11 + 9 = 31. Does the ace loop take off 10 twice? No, once is enough: 31 − 10 = 21. So `21` is printed. This is what professional programmers do: test each piece on its own.

### Version 2: the game loop rewritten (steps 6–8)

In [ ]:
def compare(user_score, computer_score):
    """Return the result message."""
    if user_score == computer_score:
        return "Draw"
    elif computer_score == 0:
        return "Lose, opponent has Blackjack"
    elif user_score == 0:
        return "Win with a Blackjack"
    elif user_score > 21:
        return "You went over. You lose"
    elif computer_score > 21:
        return "Opponent went over. You win"
    elif user_score > computer_score:
        return "You win"
    else:
        return "You lose"


user_hand = Hand()
computer_hand = Hand()
for _ in range(2):
    user_hand.add_card()
    computer_hand.add_card()

game_over = False
while not game_over:
    print(f"Your cards: {user_hand.cards}, current score: {user_hand.score()}")
    print(f"Computer's first card: {computer_hand.cards[0]}")
    if user_hand.score() == 0 or user_hand.score() > 21 or computer_hand.score() == 0:
        game_over = True
    elif input("Type 'y' to get another card, 'n' to pass: ") == "y":
        user_hand.add_card()
    else:
        game_over = True

while computer_hand.score() != 0 and computer_hand.score() < 17:
    computer_hand.add_card()

print(f"Your final hand: {user_hand.cards}, final score: {user_hand.score()}")
print(f"Computer's final hand: {computer_hand.cards}, final score: {computer_hand.score()}")
print(compare(user_hand.score(), computer_hand.score()))

> 💡 **Explanation**
>
> **Comparing the two versions (step 9):**
>
> | Version 1 (functions) | Version 2 (class) |
> |---|---|
> | `user_cards = [deal_card(), deal_card()]` | `user_hand = Hand()` + `add_card()` twice |
> | `user_cards.append(deal_card())` | `user_hand.add_card()` |
> | `calculate_score(user_cards)` | `user_hand.score()` |
> | `compare(...)` | `compare(...)`, unchanged (as step 8 says) |
>
> **No Blackjack rule has changed.** Only the place of the data and the logic has changed: now the cards and the things done to them sit **together** on one object. This is called **refactoring**: changing the structure of the code without changing what it does.
>
> **Why is it better?** If you add a third player, you just make another `Hand()`. You don't need any new lists or functions.
>
> **Stretch (step 6):** write a `Player` class that has a `name` and a `Hand` as attributes: `self.hand = Hand()`. This is called **composition**: one object inside another object.